In [8]:
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://www.jobkorea.co.kr/Search/?stext=데이터분석"
headers = {"User-Agent": "Mozilla/5.0"}
resp = requests.get(url, headers=headers, timeout=20)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")

records = []

for a in soup.select('a[data-sentry-component="Title"]'):
    title = a.get_text(strip=True)
    href = a.get("href")
    if not title or not href:
        continue
    url_abs = href if href.startswith("http") else f"https://www.jobkorea.co.kr{href}"

    card = None
    p = a
    for _ in range(8):
        p = p.parent
        if not p:
            break
        cls = " ".join(p.get("class", []))
        if "Flex_direction_column__" in cls:
            card = p
            break
    if not card:
        continue
    
    site = "Job_Korea"
    comp = card.select_one('span[data-sentry-element="Typography"][class*="Typography_variant_size16__"]')
    company = comp.get_text(strip=True) if comp else None

    cond_div = card.select_one('div[class*="Flex_gap_space16__"][class*="Flex_direction_row__"]')
    cond_list = []
    if cond_div:
        cond_list = [sp.get_text(strip=True) for sp in cond_div.select("span")]

    if company and title and url_abs:
        records.append({
            "Site" : site,
            "Col_Company": company,
            "Col_Recruit": title,
            "Col_detail": cond_list,
            "Col_url": url_abs
        })

df = pd.DataFrame(records)
display(df)

,Site,Col_Company,Col_Recruit,Col_detail,Col_url
0,Job_Korea,㈜쿡섬,"데이터분석전문가,Python,10월부터","[경력9년↑, 학력무관, 계약직 외 1, 서울 중구]",https://www.jobkorea.co.kr/Recruit/GI_Read/476...
1,Job_Korea,콘센트릭스서비스코리아,[Catalyst]데이터분석컨설턴트,"[경력5년↑, 학력무관, 정규직, 서울 강남구]",https://www.jobkorea.co.kr/Recruit/GI_Read/477...
2,Job_Korea,㈜원익피앤이,[원익PNE]데이터분석및 솔루션 담당자,"[경력3년↑, 석사↑, 정규직, 경기 수원시]",https://www.jobkorea.co.kr/Recruit/GI_Read/476...
3,Job_Korea,콘센트릭스서비스코리아,[Catalyst]데이터분석컨설턴트,"[경력5년↑, 학력무관, 정규직, 서울 강남구]",https://www.jobkorea.co.kr/Recruit/GI_Read/475...
4,Job_Korea,㈜지바이크,[지쿠] 전략마케팅팀데이터분석전문가 경력 채용,"[경력3년↑, 대졸↑, 정규직, 서울 강남구]",https://www.jobkorea.co.kr/Recruit/GI_Read/474...
5,Job_Korea,넛지헬스케어㈜,[캐시워크-병역특례]데이터분석담당 산업기능요원,"[경력무관, 학력무관, 병역특례, 서울 강남구]",https://www.jobkorea.co.kr/Recruit/GI_Read/474...
6,Job_Korea,㈜메디쿼터스,[일본:누구(nugu)] 프로덕트기획팀데이터분석,"[경력10년↑, 학력무관, 정규직, 서울 강남구]",https://www.jobkorea.co.kr/Recruit/GI_Read/476...
7,Job_Korea,㈜트리노드(TreenodInc.),[서울/경력 5년 이상]데이터분석팀데이터분석가,"[경력5년↑, 학력무관, 정규직, 서울 강남구]",https://www.jobkorea.co.kr/Recruit/GI_Read/476...
8,Job_Korea,삼양식품㈜,[삼양식품] Global 물류(관리/데이터분석),"[경력8년↑, 대졸↑, 정규직, 서울 성북구]",https://www.jobkorea.co.kr/Recruit/GI_Read/473...
9,Job_Korea,넛지헬스케어㈜,[캐시워크]데이터분석담당 채용전환형 인턴,"[신입, 대졸↑, 인턴, 서울 강남구]",https://www.jobkorea.co.kr/Recruit/GI_Read/474...


In [6]:
df.to_csv("data_tmp/data_jobkorea.csv", index=False, encoding="utf-8-sig")